# Gemini Evaluation - With Thinking Token Logging

This notebook evaluates Gemini models on the OI Benchmark and captures thinking content.

**Install:** `pip install google-genai`

**Models:**
- `gemini-2.5-pro` - With thinking mode
- `gemini-2.5-flash` - Faster, cheaper
- `gemini-3-pro-preview` - Latest preview

In [ ]:
import os
from google import genai
from google.genai import types

In [ ]:
# Configure your Gemini API key
api_key = "YOUR_GEMINI_API_KEY"
client = genai.Client(api_key=api_key)

In [ ]:
import pandas as pd
import numpy as np
import os

# Load your CSV - adjust path as needed
input_csv_path = 'Benchmark/OP_Benchmark.csv'
df = pd.read_csv(input_csv_path)

# Extract the filename without extension for the output CSV
input_filename = os.path.splitext(os.path.basename(input_csv_path))[0]

print(f"Loaded {len(df)} questions from {input_filename}")
df.head(5)

In [ ]:
import time, threading
import pandas as pd
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types

# ============ CONFIGURATION ============
model_name = "gemini-3-pro-preview"  # Options: gemini-2.5-pro, gemini-2.5-flash, gemini-3-flash-preview

# Output CSV path includes both model name and input filename
CSV_PATH = f"{model_name.replace('-', '_')}_{input_filename}.csv"

MAX_WORKERS = 4      # Tune based on your rate limit
RPM_LIMIT   = 30     # Gemini free tier: ~15 RPM, paid: adjust accordingly
SAVE_EVERY  = 5      # Controls save frequency

# Thinking configuration
ENABLE_THINKING = True  # Set to False to disable thinking mode
THINKING_BUDGET = None  # None = dynamic (default high), or set specific token count like 16384

# ============ COLUMNS FOR LOGGING ============
if 'answer_to_prompt_1' not in df.columns:
    df['answer_to_prompt_1'] = None
if 'answer_to_prompt_2' not in df.columns:
    df['answer_to_prompt_2'] = None
if 'input_tokens_1' not in df.columns:
    df['input_tokens_1'] = None
if 'input_tokens_2' not in df.columns:
    df['input_tokens_2'] = None
if 'output_tokens_1' not in df.columns:
    df['output_tokens_1'] = None
if 'output_tokens_2' not in df.columns:
    df['output_tokens_2'] = None
# Thinking/reasoning token counts
if 'thinking_tokens_1' not in df.columns:
    df['thinking_tokens_1'] = None
if 'thinking_tokens_2' not in df.columns:
    df['thinking_tokens_2'] = None
# Thinking content (thought summaries)
if 'thinking_content_1' not in df.columns:
    df['thinking_content_1'] = None
if 'thinking_content_2' not in df.columns:
    df['thinking_content_2'] = None

df.reset_index(drop=True, inplace=True)

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)

# ============ SAVE LOCK ============
save_lock = threading.Lock()

def save():
    with save_lock:
        df.to_csv(CSV_PATH, index=False)

# ============ API CALL WITH RETRY ============
def ask_until_success(prompt):
    """Returns: (answer_text, input_tokens, output_tokens, thinking_tokens, thinking_content)"""
    while True:
        try:
            rpm_limiter.wait()
            
            # Configure generation with thinking
            config = types.GenerateContentConfig(
                max_output_tokens=8192
            )
            
            if ENABLE_THINKING:
                if THINKING_BUDGET is not None:
                    config.thinking_config = types.ThinkingConfig(
                        include_thoughts=True,
                        thinking_budget=THINKING_BUDGET
                    )
                else:
                    # Dynamic thinking (model decides budget - default high)
                    config.thinking_config = types.ThinkingConfig(
                        include_thoughts=True
                    )
            
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=config
            )
            
            # Extract answer and thinking content from parts
            answer_text = ""
            thinking_content = ""
            
            if response.candidates and response.candidates[0].content.parts:
                for part in response.candidates[0].content.parts:
                    if hasattr(part, 'text') and part.text:
                        if hasattr(part, 'thought') and part.thought:
                            # This is thinking content
                            thinking_content += part.text
                        else:
                            # This is the actual answer
                            answer_text += part.text
            
            answer_text = answer_text.strip()
            thinking_content = thinking_content.strip()
            
            # Extract token counts
            input_tokens = 0
            output_tokens = 0
            thinking_tokens = 0
            
            if hasattr(response, 'usage_metadata'):
                input_tokens = getattr(response.usage_metadata, 'prompt_token_count', 0) or 0
                output_tokens = getattr(response.usage_metadata, 'candidates_token_count', 0) or 0
                thinking_tokens = getattr(response.usage_metadata, 'thoughts_token_count', 0) or 0

            if answer_text:  # Ensure valid answer
                return answer_text, input_tokens, output_tokens, thinking_tokens, thinking_content
            
            print("Empty response — retrying in 4s…")
            time.sleep(4)

        except Exception as e:
            error_msg = str(e).lower()
            
            # Handle rate limiting
            if 'quota' in error_msg or 'rate' in error_msg or '429' in error_msg:
                print(f"Rate limit hit — retrying in 10s: {e}")
                time.sleep(10)
            
            # Handle other API errors
            elif 'api' in error_msg or 'service' in error_msg:
                print(f"API issue — retrying in 4s: {e}")
                time.sleep(4)
            
            # Handle blocked content (safety filters)
            elif 'blocked' in error_msg or 'safety' in error_msg:
                print(f"Content blocked by safety filters — returning empty: {e}")
                return "[BLOCKED_BY_SAFETY_FILTER]", 0, 0, 0, ""
            
            # Unexpected errors
            else:
                print(f"Unexpected error — retrying in 4s: {e}")
                time.sleep(4)

# ============ WORKER FUNCTION ============
row_lock = threading.Lock()
progress = {"count": 0}

def process_row(i):
    # Resume skip
    if pd.notna(df.at[i, 'answer_to_prompt_1']) and pd.notna(df.at[i, 'answer_to_prompt_2']):
        return

    p1 = str(df.at[i, 'prompt.1'])
    p2 = str(df.at[i, 'prompt.2'])
    opt = str(df.at[i, 'OPTIONS'])

    if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
    if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

    a1, in1, out1, think1, think_content1 = ask_until_success(p1)
    a2, in2, out2, think2, think_content2 = ask_until_success(p2)

    with row_lock:
        df.at[i, 'answer_to_prompt_1'] = a1
        df.at[i, 'answer_to_prompt_2'] = a2
        df.at[i, 'input_tokens_1'] = in1
        df.at[i, 'input_tokens_2'] = in2
        df.at[i, 'output_tokens_1'] = out1
        df.at[i, 'output_tokens_2'] = out2
        df.at[i, 'thinking_tokens_1'] = think1
        df.at[i, 'thinking_tokens_2'] = think2
        df.at[i, 'thinking_content_1'] = think_content1
        df.at[i, 'thinking_content_2'] = think_content2
        progress["count"] += 1

        if progress["count"] % SAVE_EVERY == 0:
            save()
            print(f"Saved @ {progress['count']} rows | Thinking: {think1}, {think2} tokens")

# ============ SUBMIT WORK ============
todo = [i for i in range(len(df))
        if pd.isna(df.at[i, 'answer_to_prompt_1']) or pd.isna(df.at[i, 'answer_to_prompt_2'])]

print(f"Submitting {len(todo)} rows (out of {len(df)})")
print(f"Model: {model_name}")
print(f"Thinking: {'Enabled' if ENABLE_THINKING else 'Disabled'} (budget: {'dynamic' if THINKING_BUDGET is None else THINKING_BUDGET})")
print(f"Output: {CSV_PATH}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, i) for i in todo]
    completed = 0
    for _ in as_completed(futures):
        completed += 1
        if completed % 10 == 0:
            print(f"Progress: {completed}/{len(todo)}")

save()

# ============ FINAL VERIFICATION ============
missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()

while missing:
    print(f"Fixing {len(missing)} missing rows…")
    for i in missing:
        process_row(i)
        save()
    missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()

save()
print(f"100% COMPLETE — All rows answered and saved to {CSV_PATH}")